# PIT daily signal workflow

This notebook uses only `us_equities_datahandler` and `us_equities_signal_research`. It does not import the retired research pipeline.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import us_equities_datahandler as dh
import us_equities_signal_research as signals

dh.configure(
    crsp_ranked_daily='/datahandler/data/derived/crsp_ranked_daily',
    compustat_annual='/datahandler/data/derived/compustat/annual.parquet',
    ccm_links='/datahandler/data/derived/ccm/links.parquet',
)

START, END = '2000-01-01', '2025-12-31'
RANK_LTE = 500
SIGNAL = 'knowledge_capital_built_versus_bought'
WEIGHT_METHOD = 'zscore'       # zscore | rank | quintile
SIC_LEVEL = None               # None | 1


In [ ]:
top500 = dh.universe.market_cap(
    start=START, end=END, rank_lte=RANK_LTE, lag=1,
)
raw_signal = signals.panel(SIGNAL, data=dh, universe=top500)

# These are all daily wide panels on the same state calendar.
returns = signals.crsp_panel('ret', data=dh, universe=raw_signal.universe)
sic = signals.crsp_panel('siccd', data=dh, universe=raw_signal.universe)
groups = None if SIC_LEVEL is None else signals.sic_division_panel(sic)
weights = signals.score(
    raw_signal, universe=top500, groups=groups,
    method=WEIGHT_METHOD, winsorize=0.01,
    normalize='net_zero_gross_one',
)


In [ ]:
daily_pnl = signals.realized_returns(weights, returns)
metrics = signals.performance_metrics(daily_pnl, weights)
display(metrics)

curve = ((1 + daily_pnl.fillna(0)).cumprod() - 1) * 100
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(curve.index, curve, lw=1.25)
ax.axhline(0, color='black', lw=.7)
ax.set(title=f'{SIGNAL} | {WEIGHT_METHOD} | SIC_LEVEL={SIC_LEVEL}', ylabel='Cumulative simple return (%)')
ax.grid(alpha=.25)
plt.show()


The raw signal is known after close on *t*. `realized_returns` uses `target[t-2] × ret[t]`: the position forms at the *t+1* close and earns the *t+1*→*t+2* close-to-close return. The two state rows before `START` enable PnL on the first reported CRSP session.